# mu-logsigma-encoder-head — faded example 3: Convert logsigma output to standard deviation for sampling

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `mu-logsigma-encoder-head`. Running the beacon reports progress on the `VAE: mu+logsigma encoder head` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `VAE: mu+logsigma encoder head` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mu-logsigma-encoder-head`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mu-logsigma-encoder-head"
DD_SUBTOPIC = "VAE: mu+logsigma encoder head"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A VAE encoder outputs `logsigma` (log standard deviation) rather than the standard deviation directly. Converting back to std requires `sigma = exp(logsigma)`. Storing the log-std keeps the representation unconstrained (any real number) while ensuring the std is always positive after exponentiation — no clamping or activation needed.

## Faded exercise 3

Implement `encode_to_std(features, weight, bias, latent_dim)` that:
1. Computes `params = features @ weight.T + bias`.
2. Splits into `mu, logsigma = params.chunk(2, dim=-1)`.
3. Computes `sigma = torch.exp(logsigma)`.
4. Returns `(mu, sigma)` (NOT `logsigma`).

Your task: **fill in the chunk split and the exp conversion, returning (mu, sigma)**.

**Fill in:** Splitting params with chunk(2, dim=-1) to get (mu, logsigma), computing sigma = torch.exp(logsigma), and returning the tuple (mu, sigma).

In [ ]:
import torch

def encode_to_std(features: torch.Tensor, weight: torch.Tensor,
                  bias: torch.Tensor, latent_dim: int):
    params = features @ weight.T + bias   # (B, 2*latent_dim)
    raise NotImplementedError()  # TODO: Splitting params with chunk(2, dim=-1) to get (mu, logsigma), computing sigma = torch.exp(logsigma), and returning the tuple (mu, sigma).

def _test():
    import torch
    torch.manual_seed(0)
    B, D, L = 4, 12, 5
    features = torch.randn(B, D)
    weight   = torch.randn(2 * L, D)
    bias     = torch.randn(2 * L)
    mu, sigma = encode_to_std(features, weight, bias, L)
    assert mu.shape == (B, L)
    assert sigma.shape == (B, L)
    assert (sigma > 0).all(), "sigma must be positive after exp"
    # sigma = exp(logsigma): verify via logsigma = log(sigma)
    params = features @ weight.T + bias
    _, logsigma = params.chunk(2, dim=-1)
    assert torch.allclose(sigma, torch.exp(logsigma), atol=1e-6)


def _test():
    import torch
    torch.manual_seed(0)
    B, D, L = 4, 12, 5
    features = torch.randn(B, D)
    weight   = torch.randn(2 * L, D)
    bias     = torch.randn(2 * L)
    mu, sigma = encode_to_std(features, weight, bias, L)
    assert mu.shape == (B, L), f"mu shape: {mu.shape}"
    assert sigma.shape == (B, L), f"sigma shape: {sigma.shape}"
    # sigma must be strictly positive
    assert (sigma > 0).all(), "sigma must be positive"
    # sigma should equal exp(logsigma)
    params = features @ weight.T + bias
    _, logsigma = params.chunk(2, dim=-1)
    assert torch.allclose(sigma, torch.exp(logsigma), atol=1e-6)
    # mu should equal first half
    mu_ref = params[:, :L]
    assert torch.allclose(mu, mu_ref, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch

def encode_to_std(features: torch.Tensor, weight: torch.Tensor,
                  bias: torch.Tensor, latent_dim: int):
    params = features @ weight.T + bias
    mu, logsigma = params.chunk(2, dim=-1)
    sigma = torch.exp(logsigma)
    return mu, sigma

def _test():
    import torch
    torch.manual_seed(0)
    B, D, L = 4, 12, 5
    features = torch.randn(B, D)
    weight   = torch.randn(2 * L, D)
    bias     = torch.randn(2 * L)
    mu, sigma = encode_to_std(features, weight, bias, L)
    assert mu.shape == (B, L)
    assert sigma.shape == (B, L)
    assert (sigma > 0).all()
    params = features @ weight.T + bias
    _, logsigma = params.chunk(2, dim=-1)
    assert torch.allclose(sigma, torch.exp(logsigma), atol=1e-6)
```
</details>